# EDA: King County Housing — Client William Rodriguez

This notebook follows the same structure as the reference notebook (`detailed-exploratory-data-analysis-with-python.ipynb`), adapted to the King County housing data and fixed for the differences between that dataset and this one:

- Target column is `price`, not `SalePrice` — and `price` is the **first** column here, not the last, so anywhere the reference used positional slicing (`[:-1]`, `[0]`) assuming the target sits at the end, this version selects by column name instead.
- `house_id` / `sale_id` are identifier columns (like `Id` in the reference) — dropped from analysis, not treated as features.
- This dataset is almost entirely numeric — there's no rich set of categorical/object columns like `Neighborhood` or `MSZoning`. The categorical-vs-quantitative section is adapted to use `zipcode`, `condition`, `grade`, `waterfront`, `view`, and the `is_city`/`is_country` flags built during segmentation as the categorical stand-ins.
- `sns.distplot` is deprecated in current seaborn — replaced with `sns.histplot(..., kde=True)`.
- Positional Series indexing like `series[0]` no longer works on non-integer indexes in current pandas — replaced with `series.iloc[0]`.

**Client:** William Rodriguez, buyer, household of 2, wants a **country** home (best timing, non-renovated) and a **city** home (fast, central). See `04_eda.ipynb` for the full working assumptions.

## Preparations

Import the libraries and load the joined dataset.

In [1]:
import operator
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use("bmh")
%matplotlib inline

In [ ]:
df = pd.read_csv("../../../data/eda.csv")
df["date"] = pd.to_datetime(df["date"])
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data/eda.csv'

In [ ]:
df.info()

From these informations we can check whether some features are too sparse to be useful, the way `Alley` and `PoolQC` were in the reference dataset (91/1460 and 7/1460 non-null, respectively). Let's apply the same 30%-non-null threshold here and see what, if anything, gets dropped — and separately drop `house_id`/`sale_id`, which are identifiers, not features (the same reasoning the reference used to drop `Id`).

In [ ]:
# df.count() does not include NaN values
df2 = df[[column for column in df if df[column].count() / len(df) >= 0.3]]
df2 = df2.drop(columns=["house_id", "sale_id"])

print("List of dropped columns:", end=" ")
for c in df.columns:
    if c not in df2.columns:
        print(c, end=", ")
print("\n")
df = df2

Unlike the reference dataset, nothing here clears the 30%-missing bar — even the sparsest column (`yr_renovated`, 17.82% missing) is well within it. Only the two identifier columns get dropped. Worth remembering: `yr_renovated == 0` most likely means "never renovated," not "unknown," per `column_names.md` — so it's not really missing data in the way `Alley` was.

Now lets take a look at how the housing price is distributed

In [ ]:
print(df["price"].describe())
plt.figure(figsize=(9, 8))
sns.histplot(df["price"], color="g", bins=100, kde=True)
plt.show()

Prices are skewed right, same shape as `SalePrice` in the reference — a long tail of expensive homes above roughly $1M. A log transform (`np.log1p`) flattens this out if you need a more normal-looking distribution later.

## Numerical data distribution

First, list the dtypes present, then isolate the numeric columns.

In [ ]:
list(set(df.dtypes.tolist()))

In [ ]:
df_num = df.select_dtypes(include=["float64", "int64"])
df_num.head()

Now lets plot them all:

In [ ]:
df_num.hist(figsize=(16, 20), bins=50, xlabelsize=8, ylabelsize=8)
plt.tight_layout()

Features such as `sqft_living`, `sqft_above`, and `sqft_living15` look like they share a similar right-skewed shape to `price`. Worth investigating further below.

#### Correlation

Now find which features are strongly correlated with `price`, stored in `golden_features_list`. Reuse `df_num` for this.

Note: the reference notebook used `df_num.corr()['SalePrice'][:-1]`, relying on `SalePrice` being the last column. Here, `price` is the **first** column of `df_num`, so we drop it by name instead of slicing by position — a position-based `[:-1]` would silently do the wrong thing on this dataset.

In [ ]:
df_num_corr = df_num.corr()["price"].drop("price")
golden_features_list = df_num_corr[abs(df_num_corr) > 0.5].sort_values(ascending=False)
print(
    "There is {} strongly correlated values with price:\n{}".format(
        len(golden_features_list), golden_features_list
    )
)

This list is incomplete since correlation is affected by outliers and zero-inflated features (the same issue the reference ran into with `OpenPorchSF`). Let's plot the numerical features against `price` first, then repeat the correlation check after removing zero values from features where 0 likely means "absence" (`sqft_basement`, `waterfront`, `view`, `yr_renovated`).

In [ ]:
feature_cols = [c for c in df_num.columns if c != "price"]
for i in range(0, len(feature_cols), 5):
    sns.pairplot(data=df_num, x_vars=feature_cols[i : i + 5], y_vars=["price"])

Several features show a lot of points sitting on `x = 0` — most obviously `sqft_basement` (no basement), `waterfront` (not on the water), `view` (no view rating), and `yr_renovated` (never renovated). Same situation as `OpenPorchSF` in the reference: a house without a feature isn't the same as a house with "0 units" of it. Let's remove the zero rows and recheck correlation, one feature at a time.

In [ ]:
individual_features_df = []
for col in feature_cols:
    tmp_df = df_num[[col, "price"]]
    tmp_df = tmp_df[tmp_df[col] != 0]
    individual_features_df.append(tmp_df)

# .iloc[0] instead of [0]: current pandas no longer allows positional
# lookup on a Series with a string index via plain [] indexing.
all_correlations = {
    feature.columns[0]: feature.corr()["price"].iloc[0]
    for feature in individual_features_df
}
all_correlations = sorted(all_correlations.items(), key=operator.itemgetter(1))
for key, value in all_correlations:
    print("{:>15}: {:>15}".format(key, value))

With zero-inflated rows removed, here's the updated `golden_features_list`:

In [ ]:
golden_features_list = [key for key, value in all_correlations if abs(value) >= 0.5]
print(
    "There is {} strongly correlated values with price:\n{}".format(
        len(golden_features_list), golden_features_list
    )
)

### Conclusion so far

Correlation analysis on the numerical features points to `sqft_living`, `sqft_above`, `grade`, `sqft_living15`, `bathrooms`, and similar size/quality measures as the strongest single predictors of `price` — unsurprising, but useful to confirm before building William Rodriguez's specific recommendations on top of it.

## Feature to feature relationship

Plotting every numerical feature against every other would be unreadable. Instead, look at a correlation heatmap restricted to feature pairs, excluding `price` (already examined above).

In [ ]:
corr = df_num.drop(columns=["price"]).corr()
plt.figure(figsize=(12, 10))
sns.heatmap(
    corr[(corr >= 0.5) | (corr <= -0.4)],
    cmap="viridis",
    vmax=1.0,
    vmin=-1.0,
    linewidths=0.1,
    annot=True,
    annot_kws={"size": 8},
    square=True,
)
plt.show()

`sqft_living` correlates strongly with `sqft_above`, `grade`, `bathrooms`, and `sqft_living15` — all measures of house size/quality that naturally move together, similar to how `1stFlrSF`/`TotalBsmtSF` did in the reference. `lat`/`zipcode` relationships here are the geographic angle worth digging into separately for William Rodriguez's city/country question, since raw correlation won't capture that non-linear, location-based effect well.

## Q -> Q (Quantitative to Quantitative relationship)

Look at how the strongest numeric predictors relate to `price` individually, with a fitted regression line.

In [ ]:
features_to_analyse = golden_features_list + ["price"]
features_to_analyse

In [ ]:
fig, axes = plt.subplots(round(len(features_to_analyse) / 3), 3, figsize=(18, 12))

for i, ax in enumerate(fig.axes):
    if i < len(features_to_analyse) - 1:
        sns.regplot(x=features_to_analyse[i], y="price", data=df[features_to_analyse], ax=ax)

plt.tight_layout()

## C -> Q (Categorical to Quantitative relationship)

The reference dataset had ~39 true categorical columns (`Neighborhood`, `MSZoning`, etc.). This one doesn't — almost everything here is numeric by nature. The closest categorical stand-ins are `zipcode`, `condition`, `grade`, `waterfront`, and `view` (all effectively categories even though they're stored as numbers), plus the `is_city`/`is_country` flags built in `04_eda.ipynb`'s segmentation cell for William Rodriguez's geographic question.

If you haven't already run that segmentation cell in this kernel, run it here first so `is_city` exists:

In [ ]:
SEATTLE_DOWNTOWN = (47.6062, -122.3321)

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))

df["dist_to_downtown_km"] = haversine_km(df["lat"], df["long"], *SEATTLE_DOWNTOWN)
df["is_city"] = df["dist_to_downtown_km"] <= df["dist_to_downtown_km"].quantile(0.25)
df["is_country"] = df["sqft_lot"] >= df["sqft_lot"].quantile(0.75)

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.boxplot(x="is_city", y="price", data=df)
plt.setp(ax.artists, alpha=0.5, linewidth=2, edgecolor="k")
plt.title("Price by City vs. Country Classification")
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))
ax = sns.boxplot(x="condition", y="price", data=df)
plt.setp(ax.artists, alpha=0.5, linewidth=2, edgecolor="k")
plt.title("Price by Condition Rating")
plt.show()

In [ ]:
renovated = df["yr_renovated"].fillna(0) > 0
plt.figure(figsize=(10, 6))
ax = sns.boxplot(x=renovated, y=df["price"])
plt.setp(ax.artists, alpha=0.5, linewidth=2, edgecolor="k")
plt.xlabel("Renovated")
plt.title("Price: Renovated vs. Non-Renovated")
plt.show()

And finally, the top zip codes by house count, the closest thing to the reference's categorical `countplot` section (`Utilities`, `Heating`, etc. dominated by one category):

In [ ]:
plt.figure(figsize=(16, 5))
df["zipcode"].value_counts().head(20).plot(kind="bar")
plt.title("Top 20 Zip Codes by Sales Volume")
plt.xlabel("Zip Code")
plt.ylabel("Number of Sales")
plt.show()

### Conclusion

Same overall approach as the reference notebook: size/quality features (`sqft_living`, `grade`, `bathrooms`) are the strongest single numeric predictors of `price`, and location (`zipcode`/distance to downtown) matters in a way plain correlation doesn't capture well — which is exactly the geographic angle William Rodriguez's two-house search depends on. Feed the `is_city`/`is_country` and renovation findings from here into the 3+ insights and 3+ recommendations required for the assignment deliverable.